# 07 -- LSTM _(больше не бейслайн, не сравнивайте с предыдущим коммитом)_

Данные беру из [06_LSTM_Data_Preparation.ipynb](/notebooks/modeling/06_LSTM_Data_Preparation.ipynb), а 91 готовый snapshot-признак из [05_Data-Modeling.ipynb](/notebooks/modeling/05_Data-Modeling.ipynb).

Здесь обучаю две отдельные модели:

- **classifier** -- вероятность того, что GMV следующих 30 дней выше выбранного порога;
- **regressor** -- `log1p(GMV)` только для пользователей выше этого порога.

Обе модели получают одинаковый вход:

`90 дней -> dynamic features -> 2-layer BiLSTM` + `static features -> MLP`.

На выходе склеиваю прогнозы в log-space, т.к. итоговая метрика -- RMSLE:

$$\log(1 + \hat y) = g(p_{buy}) \cdot \max(0, \hat z_{positive}).$$

`g` подбираю на temporal validation. Hard threshold оставляю только если он улучшает RMSLE.


## Почему оставляю такую архитектуру

- **2-layer BiLSTM:** все 90 дней уже находятся до target, поэтому обратное направление leakage не создает.
- **Dropout:** регуляризую LSTM и MLP.
- **BatchNorm + LayerNorm:** нормализую входы и representation после LSTM, по моим наблюдениям, положительно влияет на метрику.
- **Static-ветка:** добавляю признаки из [05_Data-Modeling.ipynb](/notebooks/modeling/05_Data-Modeling.ipynb), чтобы модель видела RFM, окна, тренды и денежный профиль сразу.
- **AdamW / RAdam / Adam:** сравниваю на temporal CV.
- **Early stopping + ReduceLROnPlateau + gradient clipping:** не продолжаю обучение после остановки улучшения и ограничиваю градиенты.


In [16]:
from contextlib import nullcontext
from copy import deepcopy
from pathlib import Path
import gc
import json
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "lstm" / "meta.json").exists():
            return candidate
    raise FileNotFoundError("Сначала запустите 06_LSTM_Data_Preparation.ipynb")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle"
SUBMISSION_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

LABELED_CUTOFFS = META["labeled_cutoffs"]
INFERENCE_CUTOFF = META["inference_cutoff"]
BASE_SEQUENCE_FEATURES = META["base_sequence_features"]
CALENDAR_FEATURES = META["calendar_sequence_features"]
STATIC_FEATURES = META["static_features"]
STATIC_LOG_FEATURES = META["static_log_copy_features"]

BASE_INDEX = {name: i for i, name in enumerate(BASE_SEQUENCE_FEATURES)}
STATIC_INDEX = {name: i for i, name in enumerate(STATIC_FEATURES)}
STATIC_LOG_INDICES = [STATIC_INDEX[name] for name in STATIC_LOG_FEATURES]

BATCH_SIZE = 1024
NUM_WORKERS = 4 if torch.cuda.is_available() else 0
MAX_EPOCHS = 35
PATIENCE = 5
CV_MAX_EPOCHS = 8
CV_PATIENCE = 2
RANDOM_STATE = 42

MODEL_KWARGS = {
    "hidden_size": 128,
    "num_layers": 2,
    "lstm_dropout": 0.25,
    "head_dropout": 0.30,
    "bidirectional": True,
}

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print("device:", DEVICE)
print("base sequence:", len(BASE_SEQUENCE_FEATURES), "+ calendar:", len(CALENDAR_FEATURES))
print("static:", len(STATIC_FEATURES), "+ log copies:", len(STATIC_LOG_FEATURES), "+ missing masks")

device: mps
base sequence: 13 + calendar: 6
static: 95 + log copies: 51 + missing masks


## Dataset

Базовые последовательности и static-признаки лежат в [`data/lstm`](/data/lstm). Открываю их через memory map, чтобы не держать все cutoff в RAM.

Для regressor беру только пользователей с `y > positive_threshold`; весь массив при этом не копирую.


In [17]:
class HybridDataset(Dataset):
    def __init__(self, cutoff, with_target=True, positive_only=False, positive_threshold=0.0):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.static = np.load(path / "static.npy", mmap_mode="r")
        self.calendar = np.load(path / "calendar.npy", mmap_mode="r").astype(np.float32)
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None

        self.indices = np.flatnonzero(self.y > positive_threshold) if positive_only else None

    def __len__(self):
        return len(self.X) if self.indices is None else len(self.indices)

    def __getitem__(self, i):
        j = i if self.indices is None else int(self.indices[i])

        # Календарь один для всего cutoff, поэтому добавляю его только при чтении батча.
        x = np.asarray(self.X[j], dtype=np.float32)
        x = np.concatenate([x, self.calendar], axis=1)
        static = np.array(self.static[j], dtype=np.float32, copy=True)

        x = torch.from_numpy(x)
        static = torch.from_numpy(static)

        if self.y is None:
            return x, static

        y = torch.tensor(float(self.y[j]), dtype=torch.float32)
        return x, static, y


def make_loader(cutoffs, shuffle=False, with_target=True, positive_only=False, positive_threshold=0.0):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]

    dataset = ConcatDataset([
        HybridDataset(
            cutoff,
            with_target=with_target,
            positive_only=positive_only,
            positive_threshold=positive_threshold,
        )
        for cutoff in cutoffs
    ])

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        drop_last=shuffle,  # На train не оставляю последний неполный batch из-за BatchNorm.
    )

## Динамические признаки

Из 13 базовых каналов на батче добавляю:

- `has_*`;
- ratio воронки;
- rolling mean за 7/30 дней;
- rolling activity rate за 7/30 дней;
- первые разности.

Rolling считаю causal: для дня $t$ использую только дни $\le t$.


In [18]:
def causal_mean(values, window):
    # Паддинг добавляю только слева
    values = F.pad(values.unsqueeze(1), (window - 1, 0))
    return F.avg_pool1d(values, kernel_size=window, stride=1).squeeze(1)


def first_difference(values):
    result = torch.zeros_like(values)
    result[:, 1:] = values[:, 1:] - values[:, :-1]
    return result


def safe_ratio(numerator, denominator, max_value=5.0):
    ratio = numerator / denominator.clamp_min(1e-3)
    ratio = torch.where(denominator > 0, ratio, torch.zeros_like(ratio))
    return ratio.clamp(0, max_value)


def make_sequence_features(x):
    # Для ratio возвращаю счетчики из log1p в исходный масштаб.
    searches_log = x[..., BASE_INDEX["searches"]]
    search_to_cart_log = x[..., BASE_INDEX["search_to_cart"]]
    search_to_ord_log = x[..., BASE_INDEX["search_to_ord"]]
    cat_to_cart_log = x[..., BASE_INDEX["cat_to_cart"]]
    cat_to_ord_log = x[..., BASE_INDEX["cat_to_ord"]]
    to_cart_log = x[..., BASE_INDEX["to_cart"]]
    to_ord_log = x[..., BASE_INDEX["to_ord"]]
    gmv_search_log = x[..., BASE_INDEX["gmv_search"]]
    gmv_log = x[..., BASE_INDEX["gmv"]]
    active = x[..., BASE_INDEX["active"]]

    searches = torch.expm1(searches_log).clamp_min(0)
    search_to_cart = torch.expm1(search_to_cart_log).clamp_min(0)
    search_to_ord = torch.expm1(search_to_ord_log).clamp_min(0)
    cat_to_cart = torch.expm1(cat_to_cart_log).clamp_min(0)
    cat_to_ord = torch.expm1(cat_to_ord_log).clamp_min(0)
    to_cart = torch.expm1(to_cart_log).clamp_min(0)
    to_ord = torch.expm1(to_ord_log).clamp_min(0)
    gmv_search = torch.expm1(gmv_search_log).clamp_min(0)
    gmv = torch.expm1(gmv_log).clamp_min(0)

    derived = [
        (search_to_cart > 0).float(),
        (search_to_ord > 0).float(),
        (cat_to_cart > 0).float(),
        (cat_to_ord > 0).float(),
        safe_ratio(search_to_cart, searches),
        safe_ratio(search_to_ord, searches),
        safe_ratio(to_ord, to_cart),
        safe_ratio(gmv_search, gmv, max_value=1.5),
    ]

    # Rolling считаю в log-space, чтобы редкие большие значения меньше влияли на среднее.
    for values in [searches_log, to_cart_log, to_ord_log, gmv_log]:
        derived.append(causal_mean(values, 7))
        derived.append(causal_mean(values, 30))

    derived.extend([
        causal_mean(active, 7),
        causal_mean(active, 30),
        first_difference(searches_log),
        first_difference(to_ord_log),
        first_difference(gmv_log),
    ])

    derived = torch.stack(derived, dim=-1)
    return torch.cat([x, derived], dim=-1)


DERIVED_SEQUENCE_FEATURES = [
    "has_search_to_cart", "has_search_to_ord", "has_cat_to_cart", "has_cat_to_ord",
    "search_cart_ratio", "search_order_ratio", "cart_order_ratio", "search_gmv_share_daily",
    "searches_ma7", "searches_ma30",
    "to_cart_ma7", "to_cart_ma30",
    "to_ord_ma7", "to_ord_ma30",
    "gmv_ma7", "gmv_ma30",
    "active_rate7", "active_rate30",
    "searches_delta", "to_ord_delta", "gmv_delta",
]

SEQ_INPUT_SIZE = len(BASE_SEQUENCE_FEATURES) + len(CALENDAR_FEATURES) + len(DERIVED_SEQUENCE_FEATURES)
print("effective sequence channels:", SEQ_INPUT_SIZE)

effective sequence channels: 40


## Static preprocessing

К static-признакам из [Prepared_data.parquet](/data/Prepared_data.parquet) добавляю missing-mask и `log1p`-копии heavy-tail полей.

Среднее и стандартное отклонение считаю только по train-cutoff текущего fold. `NaN` заменяю train mean уже после сохранения missing-mask.


In [19]:
STATIC_LOG_INDICES_T = torch.tensor(STATIC_LOG_INDICES, dtype=torch.long)
STATIC_AUGMENTED_SIZE = len(STATIC_FEATURES) + len(STATIC_LOG_INDICES) + len(STATIC_FEATURES)
STATIC_STATS_CACHE = {}


def augment_static_numpy(raw):
    raw = np.asarray(raw, dtype=np.float32)
    missing = ~np.isfinite(raw)

    source = raw[:, STATIC_LOG_INDICES]
    logs = np.where(
        np.isfinite(source),
        np.log1p(np.clip(source, 0, None)),
        np.nan,
    ).astype(np.float32)

    return np.concatenate([raw, logs, missing.astype(np.float32)], axis=1)


def fit_static_stats(cutoffs, chunk_size=65_536):
    key = tuple(cutoffs)
    if key in STATIC_STATS_CACHE:
        return STATIC_STATS_CACHE[key]

    sums = np.zeros(STATIC_AUGMENTED_SIZE, dtype=np.float64)
    sums_sq = np.zeros(STATIC_AUGMENTED_SIZE, dtype=np.float64)
    counts = np.zeros(STATIC_AUGMENTED_SIZE, dtype=np.int64)

    for cutoff in cutoffs:
        raw = np.load(DATA_DIR / cutoff / "static.npy", mmap_mode="r")
        for start in range(0, len(raw), chunk_size):
            block = augment_static_numpy(raw[start:start + chunk_size])
            finite = np.isfinite(block)
            safe = np.where(finite, block, 0.0).astype(np.float64)

            sums += safe.sum(axis=0)
            sums_sq += (safe * safe).sum(axis=0)
            counts += finite.sum(axis=0)

    mean = sums / np.maximum(counts, 1)
    var = sums_sq / np.maximum(counts, 1) - mean * mean
    std = np.sqrt(np.maximum(var, 1e-6))

    stats = (
        torch.tensor(mean, dtype=torch.float32),
        torch.tensor(std, dtype=torch.float32),
    )
    STATIC_STATS_CACHE[key] = stats
    return stats


def normalize_static(raw, stats):
    log_indices = STATIC_LOG_INDICES_T.to(raw.device)
    source = raw.index_select(1, log_indices)
    logs = torch.where(
        torch.isfinite(source),
        torch.log1p(source.clamp_min(0)),
        torch.full_like(source, float("nan")),
    )
    missing = (~torch.isfinite(raw)).float()
    augmented = torch.cat([raw, logs, missing], dim=1)

    mean, std = stats
    mean = mean.to(raw.device, non_blocking=True)
    std = std.to(raw.device, non_blocking=True)

    # NaN заменяю train mean, после стандартизации это ровно 0.
    augmented = torch.where(torch.isfinite(augmented), augmented, mean)
    return (augmented - mean) / std


print("effective static channels:", STATIC_AUGMENTED_SIZE)

effective static channels: 241


## Модель

Sequence-ветка: небольшая проекция -> 2-layer BiLSTM -> `last hidden + mean pooling + max pooling`.

Static-ветка: обычный MLP.

После этого склеиваю обе ветки и получаю один logit/score. Для classifier и regressor архитектура одинаковая, веса разные.


In [20]:
class HybridLSTM(nn.Module):
    def __init__(
        self,
        seq_input_size,
        static_input_size,
        hidden_size=128,
        num_layers=2,
        lstm_dropout=0.25,
        head_dropout=0.30,
        bidirectional=True,
    ):
        super().__init__()
        self.bidirectional = bidirectional
        directions = 2 if bidirectional else 1

        self.sequence_bn = nn.BatchNorm1d(seq_input_size)
        self.sequence_projection = nn.Sequential(
            nn.Linear(seq_input_size, 96),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.lstm = nn.LSTM(
            input_size=96,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        lstm_out = hidden_size * directions
        pooled_size = lstm_out * 3
        self.sequence_head = nn.Sequential(
            nn.LayerNorm(pooled_size),
            nn.Linear(pooled_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
        )

        self.static_head = nn.Sequential(
            nn.BatchNorm1d(static_input_size),
            nn.Linear(static_input_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
        )

        self.fusion = nn.Sequential(
            nn.Linear(256 + 128, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(head_dropout / 2),
            nn.Linear(64, 1),
        )

    def forward(self, sequence, static):
        sequence = make_sequence_features(sequence)
        sequence = self.sequence_bn(sequence.transpose(1, 2)).transpose(1, 2)
        sequence = self.sequence_projection(sequence)

        output, (hidden, _) = self.lstm(sequence)

        if self.bidirectional:
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            last_hidden = hidden[-1]

        mean_pool = output.mean(dim=1)
        max_pool = output.amax(dim=1)
        sequence_repr = self.sequence_head(torch.cat([last_hidden, mean_pool, max_pool], dim=1))
        static_repr = self.static_head(static)

        return self.fusion(torch.cat([sequence_repr, static_repr], dim=1)).squeeze(1)


def build_model():
    model = HybridLSTM(
        seq_input_size=SEQ_INPUT_SIZE,
        static_input_size=STATIC_AUGMENTED_SIZE,
        **MODEL_KWARGS,
    )
    return model.to(DEVICE)


probe = build_model()
print("parameters:", f"{sum(p.numel() for p in probe.parameters()):,}")
del probe

parameters: 1,040,019


## Обучение

Classifier обучаю через BCE без class weights -- хочу сохранить нормальный `predict_proba`.

Regressor обучаю через MSE по `log1p(y)`, т.к. итоговая метрика -- RMSLE.


In [21]:
def make_optimizer(model, config):
    optimizers = {
        "AdamW": torch.optim.AdamW,
        "RAdam": torch.optim.RAdam,
        "Adam": torch.optim.Adam,
    }
    optimizer = optimizers[config["optimizer"]]
    return optimizer(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])


def amp_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()


def train_one_epoch(model, loader, optimizer, task, positive_threshold, static_stats, scaler):
    model.train()
    total_loss = 0.0
    total_n = 0

    for sequence, static, y in loader:
        sequence = sequence.to(DEVICE, non_blocking=True)
        static = static.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        static = normalize_static(static, static_stats)

        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            output = model(sequence, static)
            if task == "classifier":
                target = (y > positive_threshold).float()
                loss = F.binary_cross_entropy_with_logits(output, target)
            else:
                target = torch.log1p(y)
                loss = F.mse_loss(output, target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(y)
        total_n += len(y)

    return total_loss / total_n


@torch.no_grad()
def evaluate_model(model, loader, task, positive_threshold, static_stats):
    model.eval()
    total = 0.0
    total_n = 0

    for sequence, static, y in loader:
        sequence = sequence.to(DEVICE, non_blocking=True)
        static = normalize_static(static.to(DEVICE, non_blocking=True), static_stats)
        y = y.to(DEVICE, non_blocking=True)
        output = model(sequence, static)

        if task == "classifier":
            target = (y > positive_threshold).float()
            value = F.binary_cross_entropy_with_logits(output, target, reduction="sum")
        else:
            value = F.mse_loss(output, torch.log1p(y), reduction="sum")

        total += value.item()
        total_n += len(y)

    mean = total / total_n
    return mean if task == "classifier" else math.sqrt(mean)


def fit_model(
    task,
    train_cutoffs,
    val_cutoff,
    config,
    static_stats,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=RANDOM_STATE,
):
    set_seed(seed)
    threshold = config["positive_threshold"]
    positive_only = task == "regressor"

    train_loader = make_loader(
        train_cutoffs,
        shuffle=True,
        positive_only=positive_only,
        positive_threshold=threshold,
    )
    val_loader = make_loader(
        val_cutoff,
        positive_only=positive_only,
        positive_threshold=threshold,
    )

    model = build_model()
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_state = None
    best_score = np.inf
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, task, threshold, static_stats, scaler
        )
        val_score = evaluate_model(model, val_loader, task, threshold, static_stats)
        scheduler.step(val_score)

        print(
            f"{task:10s} | epoch {epoch:02d} | train={train_loss:.5f} "
            f"| val={val_score:.5f} | lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if val_score < best_score - 1e-4:
            best_score = val_score
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_epoch, best_score

## Temporal CV для optimizer и positive threshold

В качестве кросс-валидации использую expanding-window folds по понятным причинма, на каждом fold обучаю classifier + regressor и считаю RMSLE уже после soft hurdle.

Проверяю несколько optimizer и `positive_threshold = 0/10`.


In [22]:
CV_FOLDS = [
    (LABELED_CUTOFFS[:5], LABELED_CUTOFFS[5]),
    (LABELED_CUTOFFS[:6], LABELED_CUTOFFS[6]),
    (LABELED_CUTOFFS[:7], LABELED_CUTOFFS[7]),
]

OPTIMIZER_GRID = [
    {"optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-4},
    {"optimizer": "AdamW", "lr": 5e-4, "weight_decay": 5e-4},
    {"optimizer": "RAdam", "lr": 7e-4, "weight_decay": 1e-4},
    {"optimizer": "Adam",  "lr": 5e-4, "weight_decay": 1e-5},
]
POSITIVE_THRESHOLDS = [0.0, 10.0]

SEARCH_CONFIGS = [
    {**optimizer_config, "positive_threshold": threshold}
    for optimizer_config in OPTIMIZER_GRID
    for threshold in POSITIVE_THRESHOLDS
]


@torch.no_grad()
def predict_raw(model, cutoff, static_stats, with_target=True):
    dataset = HybridDataset(cutoff, with_target=with_target)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

    model.eval()
    outputs = []
    targets = []

    for batch in loader:
        if with_target:
            sequence, static, y = batch
            targets.append(y.numpy())
        else:
            sequence, static = batch

        sequence = sequence.to(DEVICE, non_blocking=True)
        static = normalize_static(static.to(DEVICE, non_blocking=True), static_stats)
        outputs.append(model(sequence, static).float().cpu().numpy())

    outputs = np.concatenate(outputs)
    y = np.concatenate(targets) if with_target else None
    return np.asarray(dataset.users), outputs, y


def rmsle_from_log_prediction(y, pred_log):
    true_log = np.log1p(y)
    pred_log = np.clip(pred_log, 0, None)
    return float(np.sqrt(np.mean((pred_log - true_log) ** 2)))


def temporal_cross_val_score(config):
    fold_scores = []

    for fold_id, (train_cutoffs, val_cutoff) in enumerate(CV_FOLDS, start=1):
        print("\n", "=" * 70)
        print("fold", fold_id, "| val:", val_cutoff, "| config:", config)
        static_stats = fit_static_stats(train_cutoffs)

        classifier, _, _ = fit_model(
            "classifier", train_cutoffs, val_cutoff, config, static_stats,
            max_epochs=CV_MAX_EPOCHS, patience=CV_PATIENCE,
        )
        regressor, _, _ = fit_model(
            "regressor", train_cutoffs, val_cutoff, config, static_stats,
            max_epochs=CV_MAX_EPOCHS, patience=CV_PATIENCE,
        )

        _, logits, y = predict_raw(classifier, val_cutoff, static_stats)
        _, reg_log, _ = predict_raw(regressor, val_cutoff, static_stats)

        probability = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
        pred_log = probability * np.clip(reg_log, 0, None)
        score = rmsle_from_log_prediction(y, pred_log)
        fold_scores.append(score)
        print("fold RMSLE:", f"{score:.6f}")

        del classifier, regressor
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return float(np.mean(fold_scores)), float(np.std(fold_scores)), fold_scores

In [23]:
cv_rows = []

for config in SEARCH_CONFIGS:
    mean_score, std_score, fold_scores = temporal_cross_val_score(config)
    cv_rows.append({
        **config,
        "mean_rmsle": mean_score,
        "std_rmsle": std_score,
        "fold_scores": fold_scores,
    })

cv_results = pd.DataFrame(cv_rows).sort_values("mean_rmsle").reset_index(drop=True)
cv_results


fold 1 | val: 2025-09-16 | config: {'optimizer': 'AdamW', 'lr': 0.001, 'weight_decay': 0.0001, 'positive_threshold': 0.0}
classifier | epoch 01 | train=0.48347 | val=0.47036 | lr=1.00e-03
classifier | epoch 02 | train=0.48083 | val=0.46865 | lr=1.00e-03
classifier | epoch 03 | train=0.48031 | val=0.46923 | lr=1.00e-03
classifier | epoch 04 | train=0.47981 | val=0.46896 | lr=5.00e-04
regressor  | epoch 01 | train=1.68585 | val=1.14020 | lr=1.00e-03
regressor  | epoch 02 | train=1.33497 | val=1.12670 | lr=1.00e-03
regressor  | epoch 03 | train=1.31740 | val=1.12344 | lr=1.00e-03
regressor  | epoch 04 | train=1.31030 | val=1.12555 | lr=1.00e-03
regressor  | epoch 05 | train=1.30430 | val=1.12522 | lr=5.00e-04
fold RMSLE: 1.723658

fold 2 | val: 2025-10-16 | config: {'optimizer': 'AdamW', 'lr': 0.001, 'weight_decay': 0.0001, 'positive_threshold': 0.0}
classifier | epoch 01 | train=0.48155 | val=0.46970 | lr=1.00e-03
classifier | epoch 02 | train=0.47902 | val=0.46960 | lr=1.00e-03
classif

,optimizer,lr,weight_decay,positive_threshold,mean_rmsle,std_rmsle,fold_scores
0,AdamW,0.0005,0.00050,0.0,1.722318,0.007557,"[1.7246204614639282, 1.7121295928955078, 1.730..."
1,Adam,0.0005,0.00001,0.0,1.722608,0.007394,"[1.7242414951324463, 1.7128467559814453, 1.730..."
2,AdamW,0.0010,0.00010,0.0,1.722931,0.007703,"[1.7236578464508057, 1.713154911994934, 1.7319..."
3,RAdam,0.0007,0.00010,0.0,1.724125,0.007556,"[1.7248657941818237, 1.7145217657089233, 1.732..."
4,AdamW,0.0005,0.00050,10.0,1.725165,0.008291,"[1.7243120670318604, 1.7154638767242432, 1.735..."
5,Adam,0.0005,0.00001,10.0,1.725374,0.007730,"[1.7253508567810059, 1.7159181833267212, 1.734..."
6,AdamW,0.0010,0.00010,10.0,1.725479,0.008971,"[1.7253252267837524, 1.71457040309906, 1.73654..."
7,RAdam,0.0007,0.00010,10.0,1.726242,0.007931,"[1.7258199453353882, 1.7167452573776245, 1.736..."


In [24]:
best_config = {
    "optimizer": cv_results.loc[0, "optimizer"],
    "lr": float(cv_results.loc[0, "lr"]),
    "weight_decay": float(cv_results.loc[0, "weight_decay"]),
    "positive_threshold": float(cv_results.loc[0, "positive_threshold"]),
}
print("best config:", best_config)

best config: {'optimizer': 'AdamW', 'lr': 0.0005, 'weight_decay': 0.0005, 'positive_threshold': 0.0}


## Validation и holdout

После CV обучаюсь на cutoff до `2025-11-15`.

На `2025-12-15` выбираю число эпох и способ склейки classifier + regressor. `2026-01-14` оставляю как один финальный holdout.


In [25]:
TRAIN_CUTOFFS = LABELED_CUTOFFS[:8]
VAL_CUTOFF = LABELED_CUTOFFS[8]
HOLDOUT_CUTOFF = LABELED_CUTOFFS[9]

static_stats = fit_static_stats(TRAIN_CUTOFFS)

classifier, classifier_best_epoch, classifier_val_bce = fit_model(
    "classifier", TRAIN_CUTOFFS, VAL_CUTOFF, best_config, static_stats,
)
regressor, regressor_best_epoch, regressor_val_rmsle_positive = fit_model(
    "regressor", TRAIN_CUTOFFS, VAL_CUTOFF, best_config, static_stats,
)

print("classifier best epoch:", classifier_best_epoch)
print("regressor best epoch:", regressor_best_epoch)

classifier | epoch 01 | train=0.47905 | val=0.47469 | lr=5.00e-04
classifier | epoch 02 | train=0.47662 | val=0.47790 | lr=5.00e-04
classifier | epoch 03 | train=0.47611 | val=0.47262 | lr=5.00e-04
classifier | epoch 04 | train=0.47567 | val=0.47475 | lr=5.00e-04
classifier | epoch 05 | train=0.47536 | val=0.47916 | lr=2.50e-04
classifier | epoch 06 | train=0.47464 | val=0.47693 | lr=2.50e-04
classifier | epoch 07 | train=0.47429 | val=0.47388 | lr=1.25e-04
classifier | epoch 08 | train=0.47387 | val=0.47453 | lr=1.25e-04
regressor  | epoch 01 | train=1.67748 | val=1.13107 | lr=5.00e-04
regressor  | epoch 02 | train=1.33003 | val=1.12691 | lr=5.00e-04
regressor  | epoch 03 | train=1.31050 | val=1.12529 | lr=5.00e-04
regressor  | epoch 04 | train=1.30326 | val=1.12719 | lr=5.00e-04
regressor  | epoch 05 | train=1.29758 | val=1.12945 | lr=2.50e-04
regressor  | epoch 06 | train=1.28989 | val=1.12344 | lr=2.50e-04
regressor  | epoch 07 | train=1.28766 | val=1.12594 | lr=2.50e-04
regressor 

## Калибровка soft hurdle

На validation подбираю:

- `temperature` -- калибровку classifier logits;
- `gamma` -- силу probability gate;
- `hard_threshold` -- обнуление совсем низких вероятностей.

`gamma = 0` проверяет вариант без classifier. Если classifier мешает RMSLE, gate сам его отключит.


In [26]:
_, val_logits, y_val = predict_raw(classifier, VAL_CUTOFF, static_stats)
_, val_reg_log, _ = predict_raw(regressor, VAL_CUTOFF, static_stats)


def tune_hurdle(y, logits, reg_log):
    rows = []
    temperatures = [0.75, 1.0, 1.25, 1.5]
    gammas = [0.0, 0.5, 0.75, 1.0, 1.25, 1.5]
    hard_thresholds = [None, 0.05, 0.10, 0.20, 0.30]

    reg_log = np.clip(reg_log, 0, None)

    for temperature in temperatures:
        probability = 1.0 / (1.0 + np.exp(-np.clip(logits / temperature, -30, 30)))

        for gamma in gammas:
            gate = np.ones_like(probability) if gamma == 0 else probability ** gamma

            for hard_threshold in hard_thresholds:
                pred_log = gate * reg_log
                if hard_threshold is not None:
                    pred_log = np.where(probability >= hard_threshold, pred_log, 0.0)

                rows.append({
                    "temperature": temperature,
                    "gamma": gamma,
                    "hard_threshold": hard_threshold,
                    "rmsle": rmsle_from_log_prediction(y, pred_log),
                })

    return pd.DataFrame(rows).sort_values("rmsle").reset_index(drop=True)


gate_results = tune_hurdle(y_val, val_logits, val_reg_log)
gate_results.head(15)

,temperature,gamma,hard_threshold,rmsle
0,1.00,1.00,NaN,1.736231
1,1.00,1.00,0.05,1.736248
2,1.00,1.00,0.10,1.737409
3,1.25,1.00,0.05,1.738369
4,1.25,1.00,NaN,1.738370
5,1.25,1.00,0.10,1.738375
6,1.25,1.00,0.20,1.742603
7,1.00,1.25,NaN,1.743499
8,1.00,1.25,0.05,1.743508
9,1.00,1.25,0.10,1.744432


In [27]:
best_row = gate_results.iloc[0]
best_gate = {
    "temperature": float(best_row["temperature"]),
    "gamma": float(best_row["gamma"]),
    "hard_threshold": None if pd.isna(best_row["hard_threshold"]) else float(best_row["hard_threshold"]),
    "rmsle": float(best_row["rmsle"]),
}
print("best gate:", best_gate)


def apply_gate(logits, reg_log, gate_config):
    temperature = gate_config["temperature"]
    gamma = gate_config["gamma"]
    hard_threshold = gate_config["hard_threshold"]

    probability = 1.0 / (1.0 + np.exp(-np.clip(logits / temperature, -30, 30)))
    gate = np.ones_like(probability) if gamma == 0 else probability ** gamma
    pred_log = gate * np.clip(reg_log, 0, None)

    if hard_threshold is not None:
        pred_log = np.where(probability >= hard_threshold, pred_log, 0.0)

    return pred_log


_, holdout_logits, y_holdout = predict_raw(classifier, HOLDOUT_CUTOFF, static_stats)
_, holdout_reg_log, _ = predict_raw(regressor, HOLDOUT_CUTOFF, static_stats)

holdout_pred_log = apply_gate(holdout_logits, holdout_reg_log, best_gate)
holdout_score = rmsle_from_log_prediction(y_holdout, holdout_pred_log)

print("validation RMSLE:", f"{best_gate['rmsle']:.6f}")
print("holdout RMSLE:", f"{holdout_score:.6f}")

best gate: {'temperature': 1.0, 'gamma': 1.0, 'hard_threshold': None, 'rmsle': 1.7362308502197266}
validation RMSLE: 1.736231
holdout RMSLE: 1.697107


## Финальное обучение

После выбора архитектуры, optimizer, threshold, числа эпох и gate обучаю classifier и regressor на всех размеченных cutoff.

Веса сохраняю в [`models/lstm_hurdle`](/models/lstm_hurdle), чтобы prediction можно было повторить без нового поиска гиперпараметров.


In [28]:
def fit_fixed_epochs(task, train_cutoffs, config, static_stats, epochs, seed=RANDOM_STATE):
    set_seed(seed)
    threshold = config["positive_threshold"]
    positive_only = task == "regressor"

    loader = make_loader(
        train_cutoffs,
        shuffle=True,
        positive_only=positive_only,
        positive_threshold=threshold,
    )

    model = build_model()
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(epochs, 1),
        eta_min=1e-6,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model, loader, optimizer, task, threshold, static_stats, scaler
        )
        scheduler.step()
        print(f"final {task:10s} | epoch {epoch:02d}/{epochs:02d} | train={train_loss:.5f}")

    return model


FINAL_TRAIN_CUTOFFS = LABELED_CUTOFFS
final_static_stats = fit_static_stats(FINAL_TRAIN_CUTOFFS)

final_classifier = fit_fixed_epochs(
    "classifier", FINAL_TRAIN_CUTOFFS, best_config, final_static_stats,
    epochs=classifier_best_epoch,
)
final_regressor = fit_fixed_epochs(
    "regressor", FINAL_TRAIN_CUTOFFS, best_config, final_static_stats,
    epochs=regressor_best_epoch,
)

# Сохраняю веса и настройки, чтобы prediction можно было повторить без нового поиска.
torch.save(final_classifier.state_dict(), MODEL_DIR / "classifier.pt")
torch.save(final_regressor.state_dict(), MODEL_DIR / "regressor.pt")
torch.save({"mean": final_static_stats[0], "std": final_static_stats[1]}, MODEL_DIR / "static_stats.pt")

run_config = {
    "model_kwargs": MODEL_KWARGS,
    "best_config": best_config,
    "best_gate": best_gate,
    "classifier_epochs": classifier_best_epoch,
    "regressor_epochs": regressor_best_epoch,
    "holdout_rmsle": holdout_score,
}
with open(MODEL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2, default=float)

print("saved models to:", MODEL_DIR)

final classifier | epoch 01/03 | train=0.47795
final classifier | epoch 02/03 | train=0.47547
final classifier | epoch 03/03 | train=0.47428
final regressor  | epoch 01/13 | train=1.59373
final regressor  | epoch 02/13 | train=1.31815
final regressor  | epoch 03/13 | train=1.30508
final regressor  | epoch 04/13 | train=1.29444
final regressor  | epoch 05/13 | train=1.28738
final regressor  | epoch 06/13 | train=1.28416
final regressor  | epoch 07/13 | train=1.28050
final regressor  | epoch 08/13 | train=1.27751
final regressor  | epoch 09/13 | train=1.27467
final regressor  | epoch 10/13 | train=1.27238
final regressor  | epoch 11/13 | train=1.27029
final regressor  | epoch 12/13 | train=1.26958
final regressor  | epoch 13/13 | train=1.26918
saved models to: /Users/pinta/Dev/E-CUP-2026/models/lstm_hurdle


## Submission

На inference получаю probability classifier и conditional log-GMV regressor, применяю выбранный gate и только после этого возвращаюсь в деньги через `expm1`.

Финальный файл сохраняю как [`submissions/lstm_hurdle.csv`](/submissions/lstm_hurdle.csv).


In [29]:
user_ids, inference_logits, _ = predict_raw(
    final_classifier, INFERENCE_CUTOFF, final_static_stats, with_target=False
)
_, inference_reg_log, _ = predict_raw(
    final_regressor, INFERENCE_CUTOFF, final_static_stats, with_target=False
)

pred_log = apply_gate(inference_logits, inference_reg_log, best_gate)
pred = np.expm1(np.clip(pred_log, 0, None))

sample = pd.read_csv(PROJECT_ROOT / "data" / "sample_submit.csv")
pred_by_user = pd.Series(pred, index=user_ids)

submission = sample.copy()
submission["predict"] = submission["user_id"].map(pred_by_user)
submission["predict"] = submission["predict"].clip(lower=0)

out_path = SUBMISSION_DIR / "lstm_hurdle.csv"
submission.to_csv(out_path, index=False)

print("saved:", out_path)
print("rows:", len(submission))
print("zero predictions:", f"{(submission['predict'] == 0).mean():.2%}")
print("mean prediction:", f"{submission['predict'].mean():.3f}")
submission.head()

saved: /Users/pinta/Dev/E-CUP-2026/submissions/lstm_hurdle.csv
rows: 250000
zero predictions: 0.00%
mean prediction: 40.872


,user_id,predict
0,2,1.940560
1,7,83.956520
2,15,6.223347
3,18,129.363602
4,23,0.396807


> TODO перписать, чет я не рассчитывал, что 0% нулей будет... TBC...